# Database Fundamentals — SQLite Demo

This notebook uses Python's built-in `sqlite3` library to demonstrate relational tables, keys, constraints, joins, aggregation, and transactions.

In [1]:
import sqlite3
import pandas as pd
con = sqlite3.connect(':memory:')
con.execute('PRAGMA foreign_keys = ON')
print(sqlite3.sqlite_version)

3.46.1


## 1. Create a small relational schema

In [2]:
con.executescript("""
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE
);
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    email TEXT UNIQUE,
    department_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);
CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits > 0)
);
CREATE TABLE enrollments (
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrolled_on TEXT,
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);
""")
print("Tables created")

Tables created


## 2. Insert sample records

In [3]:
con.executemany("INSERT INTO departments VALUES (?, ?)", [
    (1, "Data Science"), (2, "Computer Science"), (3, "Cybersecurity")
])
con.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", [
    (101, "Asha", "asha@example.com", 1),
    (102, "Ravi", "ravi@example.com", 2),
    (103, "Neha", "neha@example.com", 1),
    (104, "Kabir", "kabir@example.com", 3)
])
con.executemany("INSERT INTO courses VALUES (?, ?, ?)", [
    (10, "SQL Fundamentals", 3), (11, "Statistics", 4)
])
con.executemany("INSERT INTO enrollments VALUES (?, ?, ?)", [
    (101, 10, "2026-01-10"), (101, 11, "2026-01-12"),
    (102, 10, "2026-01-15"), (103, 11, "2026-01-18")
])
con.commit()
pd.read_sql_query("SELECT * FROM students", con)

,student_id,student_name,email,department_id
0,101,Asha,asha@example.com,1
1,102,Ravi,ravi@example.com,2
2,103,Neha,neha@example.com,1
3,104,Kabir,kabir@example.com,3


## 3. Filter and sort

In [4]:
pd.read_sql_query("""
SELECT student_id, student_name
FROM students
WHERE department_id = 1
ORDER BY student_name
""", con)

,student_id,student_name
0,101,Asha
1,103,Neha


## 4. Join related tables

In [5]:
pd.read_sql_query("""
SELECT s.student_name, d.department_name, c.course_name, e.enrolled_on
FROM enrollments AS e
JOIN students AS s ON s.student_id = e.student_id
JOIN departments AS d ON d.department_id = s.department_id
JOIN courses AS c ON c.course_id = e.course_id
ORDER BY s.student_name, c.course_name
""", con)

,student_name,department_name,course_name,enrolled_on
0,Asha,Data Science,SQL Fundamentals,2026-01-10
1,Asha,Data Science,Statistics,2026-01-12
2,Neha,Data Science,Statistics,2026-01-18
3,Ravi,Computer Science,SQL Fundamentals,2026-01-15


## 5. Aggregate with GROUP BY

In [6]:
pd.read_sql_query("""
SELECT d.department_name, COUNT(s.student_id) AS student_count
FROM departments AS d
LEFT JOIN students AS s ON s.department_id = d.department_id
GROUP BY d.department_id, d.department_name
ORDER BY student_count DESC
""", con)

,department_name,student_count
0,Data Science,2
1,Computer Science,1
2,Cybersecurity,1


## 6. Constraints protect data integrity

In [7]:
try:
    con.execute("INSERT INTO courses VALUES (?, ?, ?)", (12, "Invalid Course", 0))
except sqlite3.IntegrityError as exc:
    print(type(exc).__name__, str(exc))
con.rollback()

IntegrityError CHECK constraint failed: credits > 0


## 7. Transactions and rollback

In [8]:
con.commit()  # Finish any earlier transaction before starting this explicit one.
con.execute("BEGIN")
con.execute("INSERT INTO courses VALUES (?, ?, ?)", (13, "Temporary Course", 2))
con.rollback()
count_after_rollback = con.execute("SELECT COUNT(*) FROM courses WHERE course_id = 13").fetchone()[0]
print("Rows after rollback:", count_after_rollback)

Rows after rollback: 0


## 8. Create an index and inspect schema

In [9]:
con.execute("CREATE INDEX idx_students_department ON students(department_id)")
schema = pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'index') ORDER BY type, name", con)
schema

,name,type
0,idx_students_department,index
1,sqlite_autoindex_departments_1,index
2,sqlite_autoindex_enrollments_1,index
3,sqlite_autoindex_students_1,index
4,courses,table
5,departments,table
6,enrollments,table
7,students,table


## 9. Practice-safe parameterized query

In [10]:
department_id = 2
pd.read_sql_query(
    "SELECT student_id, student_name FROM students WHERE department_id = ?",
    con,
    params=(department_id,)
)

,student_id,student_name
0,102,Ravi


## 10. Close the connection

In [11]:
con.close()
print("SQLite connection closed.")

SQLite connection closed.
